# Модель C: скрининг-классификатор include/exclude (обучение в YC, plan §7)

bi-encoder эмбеддинг (e5-small) пары `PICO [SEP] title. abstract` + LR-голова. Цель — **recall >= 0.95** (не терять РКИ): в пайплайне работает как recall-first второе мнение поверх LLM-скрининга (`src/adapters/screen/biencoder_clf.py`, `ScreeningPort`).

Метки: `include` / `exclude`. Адаптер использует `predict_proba` + низкий порог (`threshold`) → высокий recall. **Чекпойнт:** `screening_clf.joblib` → Object Storage → `data/models/screening_clf.joblib`.

In [ ]:
# %pip install -q sentence-transformers scikit-learn joblib

In [ ]:
# Демо-сэмпл (pico_query, title+abstract, label). ЗАМЕНИТЬ на gold-разметку из data/.
SEP = " [SEP] "
rows = [
    (
        "melanoma BRAF+MEK vs BRAF overall survival",
        "Dabrafenib plus trametinib vs dabrafenib in melanoma. OS hazard ratio 0.71.",
        "include",
    ),
    (
        "melanoma BRAF+MEK vs BRAF overall survival",
        "Cost-effectiveness analysis of targeted therapy in melanoma.",
        "exclude",
    ),
    (
        "HER2+ breast cancer trastuzumab overall survival",
        "Adjuvant trastuzumab vs chemotherapy alone in HER2+ EBC. Survival improved.",
        "include",
    ),
    (
        "HER2+ breast cancer trastuzumab overall survival",
        "Review of cardiotoxicity monitoring guidelines.",
        "exclude",
    ),
]
texts = [q + SEP + doc for q, doc, _ in rows]
labels = [y for _, _, y in rows]
len(texts)

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("intfloat/multilingual-e5-small")
X = encoder.encode(texts, normalize_embeddings=True)
X.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, recall_score

# На gold: stratified hold-out; подберите порог, где recall(include) >= 0.95.
clf = LogisticRegression(max_iter=1000)
clf.fit(X, labels)
classes = list(clf.classes_)
col = classes.index("include")
proba = clf.predict_proba(X)[:, col]
threshold = 0.3  # низкий порог включения -> высокий recall (совпадает с адаптером)
pred = ["include" if p >= threshold else "exclude" for p in proba]
print("recall(include):", recall_score(labels, pred, pos_label="include"))
print(classification_report(labels, pred))

In [ ]:
import joblib

joblib.dump(clf, "screening_clf.joblib")
# classes_ включает 'include' -> адаптер берёт P(include) и сравнивает с threshold.
# -> Object Storage -> data/models/screening_clf.joblib (путь адаптера).